# Limpieza de datos

La mayoría de la gente piensa que ser científico de datos significa estar corriendo modelos avanzados de Machine Learning todo el tiempo. La realidad es muy distinta:

![Tiempo de un científico de datos](./img/datascientist_time.jpeg)

La gran mayoría del tiempo se va **limpiando y organizando** los datos con los que queremos trabajar.

Los científicos de datos utilizamos herramientas como Pandas, Numpy, Matplotlib y Seaborn para limpiar los datos con los que queremos hacer algo.

## ETL

Un tipo de tarea que realizamos con gran frecuencia los científicos de datos son los **ETL**.

![Proceso de ETL](./img/etl.png)

ETL es un acrónimo que significa Extract, Transform, Load. Es un proceso que se utiliza para extraer datos de una fuente, transformarlos en un formato que sea adecuado para el análisis y cargarlos en una base de datos o algún otro sistema de almacenamiento.

## Ejemplo práctico

Los datos que usaremos para esta limpieza y nuestro siguiente análisis son datos de incidencia delictiva en nuestro país.

La iniciativa de datos abiertos del gobierno de México nos proporciona datos de incidencia delictiva desde 2015 hasta la fecha. Los datos se actualizan todos los meses y se pueden descargar desde el siguiente enlace: https://www.gob.mx/sesnsp/acciones-y-programas/datos-abiertos-de-incidencia-delictiva

---

Como podemos ver en el portal, se proporcionan los datos tanto a nivel estatal como a nivel municipal. En este caso, utilizaremos los datos a nivel estatal.

Descarguemos los datos y guardemos el archivo CSV en la carpeta data con el nombre `datos_delitos.csv`.

---

Ahora leamos el archivo CSV y veamos cómo se ven los datos.

Primero que nada, importemos pandas

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('data/datos_delitos.csv')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf1 in position 1: invalid continuation byte

Aquí tenemos un error muy común que suele ocurrir cuando un archivo se guarda en una computadora con cierto "encoding". 

Un encoding es una tabla que relaciona un número con un carácter. Por ejemplo, en la tabla ASCII, el número 65 corresponde a la letra "A".

Si el archivo que estamos leyendo fue guardado con un encoding distinto al que pandas espera, nos arrojará un error. 

Para solucionar esto, podemos utilizar el parámetro `encoding` de la función `pd.read_csv()` y especificar el encoding correcto.

¿Pero cómo sabemos con qué encoding cuenta el archivo?

La realidad es que la gran mayoría de los archivos los encontrarán en encoding utf-8 y Pandas no va a dar ningún error. Aquí estamos teniendo este problema porque la computadora que utilizan para generar este archivo uso un encoding diferente a utf-8.

En México, por lo general, si un archivo no está en utf-8, lo más seguro es que esté en `ISO-8859-1` o `latin1`.

Intentemos cargar el archivo especificando el encoding `ISO-8859-1`.

In [4]:
df = pd.read_csv('data/datos_delitos.csv', encoding='ISO-8859-1')
df.head()

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,2,2,3,2,0.0,1.0,2.0,0.0,0.0,0.0
3,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,2,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0


Y vemos que ya podemos leer correctamente el archivo.


¿Existe alguna forma de verificar el encoding de un archivo sin tener que estar adivinando?

ChatGTP generó el siguiente código:

In [5]:
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        rawdata = f.read()
    result = chardet.detect(rawdata)
    return result

file_path = './data/datos_delitos.csv'
encoding_info = detect_encoding(file_path)
print(f"Detected encoding: {encoding_info['encoding']}")

Detected encoding: ISO-8859-1


Sigamos...

In [6]:
df.tail(3)

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
31357,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,20,22,34,35,34,37,NaN,NaN,NaN,NaN,NaN,NaN
31358,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Electorales,Electorales,Electorales,0,1,0,11,31,23,NaN,NaN,NaN,NaN,NaN,NaN
31359,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Otros delitos del Fuero Común,Otros delitos del Fuero Común,Otros delitos del Fuero Común,151,183,174,199,208,177,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31360 entries, 0 to 31359
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Año                     31360 non-null  int64  
 1   Clave_Ent               31360 non-null  int64  
 2   Entidad                 31360 non-null  object 
 3   Bien jurídico afectado  31360 non-null  object 
 4   Tipo de delito          31360 non-null  object 
 5   Subtipo de delito       31360 non-null  object 
 6   Modalidad               31360 non-null  object 
 7   Enero                   31360 non-null  int64  
 8   Febrero                 31360 non-null  int64  
 9   Marzo                   31360 non-null  int64  
 10  Abril                   31360 non-null  int64  
 11  Mayo                    31360 non-null  int64  
 12  Junio                   31360 non-null  int64  
 13  Julio                   28224 non-null  float64
 14  Agosto                  28224 non-null

Bien. Están muy bien los datos. Sin embargo, tenemos un problema con el que es muy común encontrarnos. 

Resulta que el formato en el que está el archivo es fácil entender por seres humanos:


```markdown
| Año      | Entidad  | Enero    | Febrero  | Mes X    |
|----------|----------|----------|----------|----------|
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
```

Vemos que tenemos los meses como encabezados. Es decir, el archivo está tratando cada mes como si fuera una variable. 

Nosotros como científicos de datos estamos más interesados en conjuntos de datos que no estén en este formato de "resumen" o "tabla dinámica". Para nosotros, lo ideal sería que cada mes fuera simplemente una observación más en nuestro conjunto de datos. Es decir, queremos transformar la tabla de arriba en:

```markdown
| Año      | Entidad  | Mes        |
|----------|----------|------------|
|   ...    |   ...    |   Enero    |
|   ...    |   ...    |   Febrero  |
|   ...    |   ...    |   Marzo    |
|   ...    |   ...    |   Abril    |
|   ...    |   ...    |   Mes X    |
```

A este tipo de formato le llamos "formato largo de datos". 

Haremos las siguientes limpiezas:
* Transformar los nombres de las columnas para que no tengan caracteres especiales y estén siempre en minúsculas
* Convertir el dataset a un formato de datos "largo"

In [8]:
for col in df.columns:
    print(col)

Año
Clave_Ent
Entidad
Bien jurídico afectado
Tipo de delito
Subtipo de delito
Modalidad
Enero
Febrero
Marzo
Abril
Mayo
Junio
Julio
Agosto
Septiembre
Octubre
Noviembre
Diciembre


In [9]:
len(df.columns)

19

### Limpiar nombres de columnas

In [10]:
def limpiar_columnas(df):
    columnas_limpias = []
    for col in df.columns:
        # convertir a minusculas, reemplazar espacios por guiones bajos y eliminar caracteres especiales
        col = col.lower().replace(" ", "_").replace("ñ", "ni").replace(".", "").replace("á", "a").replace("é", "e").replace("í","i").replace("ó", "o").replace("ú", "u")
        columnas_limpias.append(col)
    
    df.columns = columnas_limpias
    
    return df


In [11]:
df.head(2)

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


In [12]:
df = limpiar_columnas(df)

In [13]:
df.head(3)

,anio,clave_ent,entidad,bien_juridico_afectado,tipo_de_delito,subtipo_de_delito,modalidad,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,2,2,3,2,0.0,1.0,2.0,0.0,0.0,0.0


Muy bien. Ahora lo que queremos hacer es quitar algunas columnas. Nos interesan nada más las siguientes:

In [14]:
df[['anio', 'entidad']]

,anio,entidad
0,2015,Aguascalientes
1,2015,Aguascalientes
2,2015,Aguascalientes
3,2015,Aguascalientes
4,2015,Aguascalientes
...,...,...
31355,2024,Zacatecas
31356,2024,Zacatecas
31357,2024,Zacatecas
31358,2024,Zacatecas


In [15]:
df = df[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'subtipo_de_delito', 'modalidad','enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']]
df.head(2)

,anio,clave_ent,entidad,tipo_de_delito,subtipo_de_delito,modalidad,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


### Formato largo de datos

Ahora, usaremos el método `melt` para convertir las columnas a observaciones.

Queremos convervar las coumnas:
* anio
* clave_ent
* entidad
* tipo_de_delito
* subtipo_de_delito
* modalidad

El resto de las columnas las vamos a juntar en una nueva columna llamada "nombre_mes" y sus valores los vamos a sumar en otra llamada "frecuencia"

In [16]:
print("Shape ", df.shape)

Shape  (31360, 18)


In [17]:
datos_long = df.melt(id_vars=['anio', 'clave_ent', 'entidad','tipo_de_delito', 'subtipo_de_delito', 'modalidad'], var_name='nombre_mes', value_name='frecuencia')

In [18]:
print("Shape: ", datos_long.shape)

Shape:  (376320, 8)


In [19]:
datos_long.head(5)

,anio,clave_ent,entidad,tipo_de_delito,subtipo_de_delito,modalidad,nombre_mes,frecuencia
0,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma de fuego,enero,3.0
1,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma blanca,enero,1.0
2,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con otro elemento,enero,0.0
3,2015,1,Aguascalientes,Homicidio,Homicidio doloso,No especificado,enero,2.0
4,2015,1,Aguascalientes,Homicidio,Homicidio culposo,Con arma de fuego,enero,0.0


In [20]:
datos_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 376320 entries, 0 to 376319
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   anio               376320 non-null  int64  
 1   clave_ent          376320 non-null  int64  
 2   entidad            376320 non-null  object 
 3   tipo_de_delito     376320 non-null  object 
 4   subtipo_de_delito  376320 non-null  object 
 5   modalidad          376320 non-null  object 
 6   nombre_mes         376320 non-null  object 
 7   frecuencia         357504 non-null  float64
dtypes: float64(1), int64(2), object(5)
memory usage: 23.0+ MB


Supongamos que para este análisis, no nos importan los niveles subtipo de delito y modalidad. O sea, no queremos tener la distinción entre homicidios dolosos y culposos (sé que son bastante diferentes, pero simplifiquemos nuestro ejemplo).

Vamos a agrupar nuestro dataframe por anio, clave_ent, entidad, tipo_de_delito y nombre_mes. Esto hará que todos los tipos de homicidios se sumen al tipo "homicidio" o todos los tipos de robo de vehículo (con o sin violencia) se sumen a "robo de vehículo".

In [21]:
datos_long = datos_long.groupby(['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes'])['frecuencia'].sum().reset_index()

In [22]:
datos_long[datos_long.tipo_de_delito == 'Robo'].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
32069,2017,3,Baja California Sur,Robo,julio,919.0
102627,2021,22,Querétaro,Robo,enero,1740.0
114150,2022,14,Jalisco,Robo,junio,3941.0
103105,2021,23,Quintana Roo,Robo,agosto,1346.0
17193,2016,4,Campeche,Robo,noviembre,87.0
137668,2023,31,Yucatán,Robo,febrero,48.0
135747,2023,27,Tabasco,Robo,enero,846.0
82469,2020,12,Guerrero,Robo,julio,450.0
97834,2021,12,Guerrero,Robo,octubre,551.0
47430,2018,3,Baja California Sur,Robo,junio,888.0


Mostremos todos los estados y su respectiva clave

In [23]:
datos_long[['clave_ent', 'entidad']].drop_duplicates().sort_values('entidad')

,clave_ent,entidad
0,1,Aguascalientes
480,2,Baja California
960,3,Baja California Sur
1440,4,Campeche
2880,7,Chiapas
3360,8,Chihuahua
3840,9,Ciudad de México
1920,5,Coahuila de Zaragoza
2400,6,Colima
4320,10,Durango


Ahora Veamos todos los datos de delitos de una entidad en específico. Por ejemplo, Nuevo león

In [26]:
datos_long[datos_long['clave_ent'] == 19].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
70239,2019,19,Nuevo León,Extorsión,enero,40.0
146953,2024,19,Nuevo León,Contra el medio ambiente,agosto,0.0
146919,2024,19,Nuevo León,Acoso sexual,enero,42.0
8875,2015,19,Nuevo León,Hostigamiento sexual,marzo,3.0
85656,2020,19,Nuevo León,Homicidio,abril,118.0
9074,2015,19,Nuevo León,Violación equiparada,diciembre,10.0
70132,2019,19,Nuevo León,Allanamiento de morada,febrero,31.0
54989,2018,19,Nuevo León,Lesiones,julio,690.0
131967,2023,19,Nuevo León,Violación simple,enero,86.0
85779,2020,19,Nuevo León,Otros delitos que atentan contra la libertad p...,enero,180.0


In [27]:
datos_long[(datos_long['clave_ent'] > 19) &  (datos_long['clave_ent'] < 24) & (datos_long['tipo_de_delito'] == 'Homicidio')].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
10306,2015,22,Querétaro,Homicidio,octubre,38.0
25177,2016,21,Puebla,Homicidio,agosto,96.0
117822,2022,22,Querétaro,Homicidio,junio,35.0
10779,2015,23,Quintana Roo,Homicidio,enero,38.0
25656,2016,22,Querétaro,Homicidio,abril,36.0
56383,2018,22,Querétaro,Homicidio,marzo,45.0
87576,2020,23,Quintana Roo,Homicidio,abril,98.0
55898,2018,21,Puebla,Homicidio,diciembre,184.0
86136,2020,20,Oaxaca,Homicidio,abril,150.0
72224,2019,23,Quintana Roo,Homicidio,mayo,110.0


### Valores de fechas

Finalmente, queremos tener una columna "fecha". Actualmente tenemos el año y el nombre del mes, pero no tenemos como tal una columna que tenga un tipo de dato fecha. Eso hace que filtrar por fecha sea complicado.

Por ejemplo, si queremos conocer todos los homicidios de Oaxaca en enero 2024, haríamos lo siguiente:

In [28]:
datos_long[
    (datos_long['clave_ent'] == 20) &
    (datos_long['tipo_de_delito'] == 'Homicidio') &
    (datos_long['anio'] == 2024) &
    (datos_long['nombre_mes'] == 'enero') 
]

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
147579,2024,20,Oaxaca,Homicidio,enero,157.0


Creemos una columna de fecha. 

Primero tenemos que convertir el nombre de mes a un número, en donde 1 es enero, 2 febrero, etc.

In [29]:
datos_long.sample(2)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
110349,2022,6,Colima,Tráfico de menores,noviembre,0.0
140693,2024,6,Colima,Allanamiento de morada,julio,0.0


In [30]:
datos_long["nueva_columna"] = "dato vacío"
datos_long.sample(6)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna
101346,2021,20,Oaxaca,Amenazas,junio,381.0,dato vacío
128550,2023,12,Guerrero,Robo,junio,427.0,dato vacío
143402,2024,11,Guanajuato,Otros delitos que atentan contra la vida y la ...,diciembre,0.0,dato vacío
85697,2020,19,Nuevo León,Incumplimiento de obligaciones de asistencia f...,julio,26.0,dato vacío
152715,2024,31,Yucatán,Contra el medio ambiente,enero,0.0,dato vacío
6089,2015,13,Hidalgo,Otros delitos del Fuero Común,julio,178.0,dato vacío


In [31]:
# Diccionario de ayuda para convertir
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

datos_long['mes'] = datos_long['nombre_mes'].map(meses)
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes
44609,2017,29,Tlaxcala,Violación simple,julio,0.0,dato vacío,7
125831,2023,7,Chiapas,Amenazas,septiembre,112.0,dato vacío,9
119139,2022,25,Sinaloa,Daño a la propiedad,enero,188.0,dato vacío,1
116615,2022,19,Nuevo León,Violación simple,septiembre,92.0,dato vacío,9
15022,2015,32,Zacatecas,Electorales,octubre,0.0,dato vacío,10


In [32]:
datos_long["frecuencia_mas_10"] = datos_long["frecuencia"] + 10
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10
41897,2017,24,San Luis Potosí,Electorales,julio,1.0,dato vacío,7,11.0
122461,2022,32,Zacatecas,Amenazas,agosto,122.0,dato vacío,8,132.0
41074,2017,22,Querétaro,Lesiones,octubre,471.0,dato vacío,10,481.0
66551,2019,11,Guanajuato,Otros delitos contra la familia,septiembre,3.0,dato vacío,9,13.0
144944,2024,14,Jalisco,Violencia de género en todas sus modalidades d...,mayo,0.0,dato vacío,5,10.0


In [33]:
datos_long["anio_mes"] = datos_long["anio"].astype(str) + datos_long["mes"].astype(str)
datos_long.sample(6)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10,anio_mes
118948,2022,24,San Luis Potosí,Robo,febrero,1179.0,dato vacío,2,1189.0,20222
54048,2018,17,Morelos,Otros delitos contra el patrimonio,abril,16.0,dato vacío,4,26.0,20184
139978,2024,4,Campeche,Otros delitos contra el patrimonio,octubre,0.0,dato vacío,10,10.0,202410
51216,2018,11,Guanajuato,Otros delitos que atentan contra la libertad p...,abril,0.0,dato vacío,4,10.0,20184
134699,2023,25,Sinaloa,Otros delitos contra el patrimonio,septiembre,2.0,dato vacío,9,12.0,20239
51024,2018,11,Guanajuato,Evasión de presos,abril,0.0,dato vacío,4,10.0,20184


In [34]:
# yyyy-mm-dd, yy-mm-dd, yymmdd, yyyy/dd/mm

In [35]:
# Agregamos la columna de fecha juntando el año y el mes
datos_long['fecha'] = pd.to_datetime(datos_long['anio'].astype(str) + datos_long['mes'].astype(str), format='%Y%m')
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10,anio_mes,fecha
133064,2023,22,Querétaro,Daño a la propiedad,mayo,148.0,dato vacío,5,158.0,20235,2023-05-01
73826,2019,26,Sonora,Robo,diciembre,859.0,dato vacío,12,869.0,201912,2019-12-01
109423,2022,4,Campeche,Violencia de género en todas sus modalidades d...,marzo,0.0,dato vacío,3,10.0,20223,2022-03-01
73420,2019,25,Sinaloa,Violencia de género en todas sus modalidades d...,febrero,0.0,dato vacío,2,10.0,20192,2019-02-01
47701,2018,4,Campeche,Falsificación,agosto,0.0,dato vacío,8,10.0,20188,2018-08-01


In [36]:
# Eliminamos las columnas que ya no necesitamos
datos_long = datos_long.drop(columns=['nueva_columna'])
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,frecuencia_mas_10,anio_mes,fecha
50955,2018,11,Guanajuato,Contra el medio ambiente,enero,1.0,1,11.0,20181,2018-01-01
30515,2016,32,Zacatecas,Lesiones,septiembre,131.0,9,141.0,20169,2016-09-01
146143,2024,17,Morelos,Homicidio,marzo,144.0,3,154.0,20243,2024-03-01
55484,2018,20,Oaxaca,Narcomenudeo,mayo,28.0,5,38.0,20185,2018-05-01
114430,2022,15,México,Falsificación,octubre,245.0,10,255.0,202210,2022-10-01


Veamos los homicidios en oaxaca de enero 2024 a la fecha

In [37]:
datos_long[
    (datos_long.tipo_de_delito == "Homicidio") &
    (datos_long.clave_ent == 20) &
    (datos_long.fecha >= '2024-01-01')
].sort_values('fecha')

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,frecuencia_mas_10,anio_mes,fecha
147579,2024,20,Oaxaca,Homicidio,enero,157.0,1,167.0,20241,2024-01-01
147580,2024,20,Oaxaca,Homicidio,febrero,170.0,2,180.0,20242,2024-02-01
147583,2024,20,Oaxaca,Homicidio,marzo,190.0,3,200.0,20243,2024-03-01
147576,2024,20,Oaxaca,Homicidio,abril,208.0,4,218.0,20244,2024-04-01
147584,2024,20,Oaxaca,Homicidio,mayo,236.0,5,246.0,20245,2024-05-01
147582,2024,20,Oaxaca,Homicidio,junio,202.0,6,212.0,20246,2024-06-01
147581,2024,20,Oaxaca,Homicidio,julio,0.0,7,10.0,20247,2024-07-01
147577,2024,20,Oaxaca,Homicidio,agosto,0.0,8,10.0,20248,2024-08-01
147587,2024,20,Oaxaca,Homicidio,septiembre,0.0,9,10.0,20249,2024-09-01
147586,2024,20,Oaxaca,Homicidio,octubre,0.0,10,10.0,202410,2024-10-01


In [38]:
datos_finales = datos_long[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes', 'fecha', 'frecuencia']]
datos_finales.head(2)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,fecha,frecuencia
0,2015,1,Aguascalientes,Aborto,abril,2015-04-01,0.0
1,2015,1,Aguascalientes,Aborto,agosto,2015-08-01,0.0


Ya que tenemos muestros datos bien estructurados, los podemos guardar en nuestra computadora. Los guardaremos con el nombre "delitos.csv"

In [39]:
datos_finales.to_csv('data/delitos.csv', index=False)